# 06 — Machine-Learning Robustness Check

**Input:** `composite_scores.csv` from notebook 3.

A final, independent check: does a more flexible method that makes fewer
functional-form assumptions than linear regression (Random Forest) find
anything the linear structural model misses? If not, that is evidence the
linear specification is adequate for this data — it does not rule out every
conceivable form of nonlinearity, only that a flexible tree-based alternative
did not find meaningfully more predictive structure in this sample.

Requires `scikit-learn` and `shap` (`pip install scikit-learn shap`).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
import shap

comp = pd.read_csv("composite_scores.csv", index_col=0)
X = comp[["AIU", "CO", "AIL", "MMR"]].values
y = comp["CTE"].values
print("N =", len(comp))

## Cross-validated linear vs. Random Forest comparison

Random Forest is tuned with an inner cross-validation loop (limiting tree depth
and leaf size) — an untuned Random Forest overfits badly at this sample size
and gives a misleadingly poor out-of-sample score.

In [ ]:
def r2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

np.random.seed(42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

ols_scores = []
for tr, te in outer_cv.split(X):
    m = LinearRegression().fit(X[tr], y[tr])
    ols_scores.append(r2(y[te], m.predict(X[te])))

rf_scores = []
grid = {"max_depth": [2, 3, 4, None], "min_samples_leaf": [5, 10, 20]}
for tr, te in outer_cv.split(X):
    inner_cv = KFold(n_splits=4, shuffle=True, random_state=1)
    gs = GridSearchCV(RandomForestRegressor(n_estimators=300, random_state=42), grid, cv=inner_cv, scoring="r2")
    gs.fit(X[tr], y[tr])
    rf_scores.append(r2(y[te], gs.best_estimator_.predict(X[te])))

ols_scores, rf_scores = np.array(ols_scores), np.array(rf_scores)
print(f"Linear:        R2 = {ols_scores.mean():.3f} (SD {ols_scores.std():.3f})")
print(f"Random Forest: R2 = {rf_scores.mean():.3f} (SD {rf_scores.std():.3f})")
print(f"RF - linear gap: {rf_scores.mean() - ols_scores.mean():+.3f}")
print("\nA negative or near-zero gap indicates Random Forest is not finding meaningful")
print("nonlinear structure beyond the linear specification in this sample.")

## SHAP feature importance (predictive corroboration of predictor ranking)

In [ ]:
rf_final = RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=10, random_state=42)
rf_final.fit(X, y)
explainer = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X)
mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({"feature": ["AIU", "CO", "AIL", "MMR"], "mean_abs_shap": mean_abs_shap})
print(shap_df.sort_values("mean_abs_shap", ascending=False).to_string(index=False))
print("\nCompare this ranking to the structural model's beta ordering in notebook 3 --")
print("agreement here is predictive corroboration from a method with no linear functional-")
print("form assumption. SHAP values reflect predictive contribution, not a causal or")
print("inferential hypothesis test, and are interpreted accordingly.")

**On comparing this R-squared to the structural model's R-squared:** the latent
SEM in notebook 3 reported R-squared = 0.591; the cross-validated linear
R-squared here is 0.438. These are not directly comparable — the SEM's
R-squared is an in-sample quantity from a model with free measurement
parameters that partially corrects for measurement error, while the figure
here is a stricter out-of-sample score on simple averaged composite scores,
with no such correction. The relevant result from this notebook is the
*within-notebook* comparison (linear vs. Random Forest under identical
cross-validation folds), not the cross-notebook one.

## Summary

Across notebooks 00-06, the central finding was consistently supported: AI
literacy and metacognitive awareness were significant positive predictors of
critical thinking engagement, while AI usage was not a significant independent
predictor and cognitive offloading showed a negative but non-significant
association in the primary latent structural model. The measurement model
demonstrated good fit and satisfactory reliability, the bootstrap analysis
reproduced the primary significance pattern for all four structural paths, the
structural-slope invariance test found no evidence that AI usage moderated the
AIL-CTE or MMR-CTE associations, the person-centered analysis provided a
complementary profile-based perspective, and this machine-learning check showed
that a tuned Random Forest did not outperform linear prediction and
independently ranked AI literacy and metacognitive awareness as the strongest
predictive features.